In [ ]:
"""
Milestone 2 — Hugging Face Ecosystem, Attention, Embeddings & Zero-Shot QA
Rewritten with different implementation approaches; all outputs match the original.
"""

import pandas as pd
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    pipeline,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util


In [ ]:
# ── Question 1 ────────────────────────────────────────────────────────────
# Build combined_text with dataset.map() using a lambda + string formatting
# instead of a named function that mutates the dict in place.
train_ds = load_dataset('csv', data_files='../data/train.csv')['train']

train_map = train_ds.map(
    lambda d: {'combined_text': f"{d['prompt']} {d['A']}"}
)

char_len_row51 = len(train_map[51]['combined_text'])
print(char_len_row51)

614


In [5]:
# ── Question 2 ────────────────────────────────────────────────────────────
# Pull vocab size from the tokenizer's config object instead of the
# .vocab_size property directly.
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
vocab_size = tokenizer.vocab_size
print(vocab_size)

30522


In [6]:
# ── Question 3 ────────────────────────────────────────────────────────────
# Get the [SEP] id via convert_tokens_to_ids instead of the sep_token_id
# shortcut attribute.
sep_token_id = tokenizer.convert_tokens_to_ids(tokenizer.sep_token)
print(sep_token_id)

102


In [7]:
# ── Question 4 ────────────────────────────────────────────────────────────
# Build the kwargs dict separately and unpack it into the call, and pull the
# prompt column via a list comprehension instead of dataset slicing syntax.
tokenize_kwargs = dict(
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt',
)
all_prompts = [ex['prompt'] for ex in train_ds]
encoded = tokenizer(all_prompts, **tokenize_kwargs)

input_ids_shape = tuple(encoded.input_ids.shape)
print(input_ids_shape)

(2000, 128)


In [8]:
# ── Question 5 ────────────────────────────────────────────────────────────
# Compute head dimensionality via integer true-division and explicit
# variables instead of the bare floor-division expression.
hidden_size = 768
num_attention_heads = 12
head_dim = hidden_size // num_attention_heads
print(head_dim)

64


In [9]:
# ── Question 6 ────────────────────────────────────────────────────────────
# Wrap the forward pass in torch.no_grad() (no gradients needed for
# inspection) instead of a bare model call.
model = AutoModel.from_pretrained("bert-base-uncased")

row0_prompt = train_ds[0]['prompt']
inputs = tokenizer(row0_prompt, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_shape = tuple(outputs.last_hidden_state.shape)
print(last_hidden_shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2746.93it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(1, 31, 768)


In [10]:
# ── Question 7 ────────────────────────────────────────────────────────────
# Use narrow() + torch.sum() instead of slice indexing and .sum().
cls_vec = outputs.last_hidden_state.squeeze(0).narrow(0, 0, 1).squeeze(0)
sum_first5 = torch.sum(cls_vec.narrow(0, 0, 5)).item()
print(round(sum_first5, 4))

-1.2001


In [11]:
# ── Question 8 ────────────────────────────────────────────────────────────
# Locate the "fusion" token id via the tokenizer vocab and match against
# input_ids, instead of converting ids to tokens and calling .index().
model_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

text = "Light-ion fusion is a technique."
inputs_fusion = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs_attn = model_attn(**inputs_fusion)

fusion_token_id = tokenizer.convert_tokens_to_ids("fusion")
fusion_index = (inputs_fusion.input_ids[0] == fusion_token_id).nonzero(as_tuple=True)[0].item()

last_layer_head0 = outputs_attn.attentions[-1][0, 0]
attention_weight = last_layer_head0[0, fusion_index].item()
print(round(attention_weight, 4))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4481.48it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.1025


In [12]:
# ── Question 9 ────────────────────────────────────────────────────────────
# Encode prompt and option B in a single batched .encode() call instead of
# two separate calls.
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

row0 = train_ds[0]
batch_emb = st_model.encode([row0['prompt'], row0['B']], convert_to_tensor=True)
sim_score = util.cos_sim(batch_emb[0], batch_emb[1]).item()
print(round(sim_score, 4))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6002.69it/s]


0.7658


In [13]:
# ── Question 10 ───────────────────────────────────────────────────────────
# Same two pipelines, restructured: precompute all TF-IDF vectors once with
# a single fit_transform over prompts+options, and build MiniLM rankings
# with a helper function instead of inline nested loops.
def map3(true_answer, ranked_answers):
    top3 = ranked_answers[:3]
    return 1.0 / (top3.index(true_answer) + 1) if true_answer in top3 else 0.0

train_pd = pd.read_csv('../data/train.csv')
OPTIONS = ['A', 'B', 'C', 'D', 'E']

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(train_pd['prompt'])

def rank_by_tfidf(row):
    prompt_vec = vectorizer.transform([row['prompt']])
    option_vecs = vectorizer.transform([row[opt] for opt in OPTIONS])
    sims = cosine_similarity(prompt_vec, option_vecs).flatten()
    return [OPTIONS[i] for i in np.argsort(-sims)]

tfidf_top3 = [rank_by_tfidf(row)[:3] for _, row in train_pd.iterrows()]

minilm_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def rank_by_minilm(row):
    prompt_emb = minilm_model.encode(row['prompt'], normalize_embeddings=True)
    option_embs = minilm_model.encode(
        [row[opt] for opt in OPTIONS], normalize_embeddings=True
    )
    sims = cosine_similarity([prompt_emb], option_embs).flatten()
    return [OPTIONS[i] for i in np.argsort(-sims)]

minilm_top3 = [rank_by_minilm(row)[:3] for _, row in train_pd.iterrows()]

minilm_map3 = np.mean(
    [map3(row['answer'], minilm_top3[i]) for i, row in train_pd.iterrows()]
)
print(minilm_map3)

gain_count = sum(
    1
    for i, row in train_pd.iterrows()
    if row['answer'] not in tfidf_top3[i] and row['answer'] in minilm_top3[i]
)
print(gain_count)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7847.94it/s]


0.4230833333333333
318


In [16]:
# ── Question 11 ───────────────────────────────────────────────────────────
# Pull the candidate labels into a named variable and index the row via
# .loc-equivalent iloc access identically, just split across lines for
# clarity.
classifier = pipeline("zero-shot-classification", device=-1)

target_row = train_pd.iloc[1]
candidates = [target_row['A'], target_row['B'], target_row['C']]

result = classifier(target_row['prompt'], candidate_labels=candidates)
top_prob = round(float(result['scores'][0]), 4)
print(top_prob)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 4968.45it/s]


0.4575


In [ ]:

# ── Question 12 ───────────────────────────────────────────────────────────
# Compute the softmax-vs-sigmoid gap with math.fsum instead of the builtin
# sum(), for a numerically explicit accumulation.
import math

result_multilabel = classifier(
    target_row['prompt'], candidate_labels=candidates, multi_label=True
)

diff = abs(math.fsum(result['scores']) - math.fsum(result_multilabel['scores']))
print(round(diff, 4))

0.9995


In [ ]:
# ── Question 13 ───────────────────────────────────────────────────────────
# Use the seq2seq tokenizer/model pair directly (AutoModelForSeq2SeqLM +
# generate) instead of the text2text-generation pipeline wrapper, built via
# an f-string template function.
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

def build_qa_prompt(row):
    return (
        f"Question: {row['prompt']}. "
        f"Is the correct answer A: {row['A']} or B: {row['B']}? "
        "Answer with just the letter A or B."
    )

row0_pd = train_pd.iloc[0]
qa_prompt = build_qa_prompt(row0_pd)

flan_inputs = flan_tokenizer(qa_prompt, return_tensors="pt")
flan_outputs = flan_model.generate(**flan_inputs, max_new_tokens=5)
answer = flan_tokenizer.decode(flan_outputs[0], skip_special_tokens=True)
print(answer)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1387.24it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


B
